# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [2]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [3]:
# load_dotenv(override=True)
# openai_api_key = os.getenv('OPENAI_API_KEY')
# anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
# google_api_key = os.getenv('GOOGLE_API_KEY')
# grok_api_key = os.getenv('GROK_API_KEY')

# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
# if anthropic_api_key:
#     print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
# else:
#     print("Anthropic API Key not set (and this is optional)")

# if google_api_key:
#     print(f"Google API Key exists and begins {google_api_key[:2]}")
# else:
#     print("Google API Key not set (and this is optional)")

# if grok_api_key:
#     print(f"Grok API Key exists and begins {grok_api_key[:4]}")
# else:
#     print("Grok API Key not set (and this is optional)")

In [4]:
# Connect to client libraries

# openai = OpenAI()

MODEL = "gpt-oss:20b"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
gemini = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
grok = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [5]:
OPENAI_MODEL = "gpt-oss:20b"
CLAUDE_MODEL = "gpt-oss:20b"
GROK_MODEL = "gpt-oss:20b"
GEMINI_MODEL = "gpt-oss:20b"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Linux',
  'arch': 'x86_64',
  'release': '6.8.0-94-generic',
  'version': '#96~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Jan 16 13:19:05 UTC 2',
  'kernel': '6.8.0-94-generic',
  'distro': {'name': 'Ubuntu 22.04.5 LTS', 'version': '22.04'},
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-linux-gnu'},
 'package_managers': ['apt'],
 'cpu': {'brand': 'Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz',
  'cores_logical': 64,
  'cores_physical': 32,
  'simd': ['AVX', 'AVX2', 'AVX512F', 'FMA', 'SSE4_2']},
 'toolchain': {'compilers': {'gcc': 'gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0',
   'g++': 'g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.3'},
  'linkers': {'ld_lld': ''}}}

In [7]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

**Short answer**

You **already have a working C++ compiler** (`g++ 11.4`).  
You can compile and run `main.cpp` in a single script like this:

```python
import subprocess

# 1) Compile – fastest‑time‑optimised build
compile_command = [
    "g++",
    "-std=c++20",          # or -std=c++23 if you need it
    "-O3",                 # maximum optimisation
    "-march=native",       # use all CPU‑specific instructions
    "-flto",               # link‑time optimisation
    "-pipe",               # use pipes instead of temp files
    "-s",                  # strip symbols (smaller binary, no runtime penalty)
    "main.cpp",            # source file
    "-o", "main"           # output executable name
]

compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True
)

# 2) Execute the binary
run_command = ["./main"]          # or ["./main", "<args>"] if you need arguments

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)

print("Program output:")
print(run_result.stdout)
```

**Why this works**

| Option | What it does | Why it helps the *runtime* |
|--------|--------------|----------------------------|
| `-O3` | Highest optimisation level | Generates faster code (though it may increase compile time) |
| `-march=native` | Enable all instruction set extensions available on your CPU (AVX2, AVX512F, FMA, …) | Lets the compiler use the fastest math/logic instructions the hardware supports |
| `-flto` | Link‑time optimisation | Allows the compiler to optimise across translation units in a single pass → leaner, faster binaries |
| `-pipe` | Use pipes instead of temporary files for `gcc` / `g++` | Minor compile‑time improvement, no effect on runtime |
| `-s` | Strip debugging symbols from the executable | Reduces binary size (less memory mapped) – no runtime penalty |
| `-std=c++20` (or `c++23`) | Language standard | Ensures you get the latest language features that can help write more efficient code (concepts, ranges, etc.) |

If you prefer a **single command** from a shell, you could also run:

```bash
g++ -std=c++20 -O3 -march=native -flto -pipe -s main.cpp -o main && ./main
```

but the Python snippet above is safer (no shell expansion, easier to debug, captures `stdout` and `stderr` for you).

**No additional installation needed**

Your system report shows:

```text
toolchain: {
    compilers: {
        gcc:   'gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0',
        g++:   'g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0',
    }
}
```

So `g++` is present in `/usr/bin/g++`.  If you ever need to confirm:

```bash
which g++            # → /usr/bin/g++
g++ --version        # → g++ (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
```

If, for some reason, `g++` were missing, you could install it with:

```bash
sudo apt update
sudo apt install g++   # pulls in all prerequisites
```

But in your situation, the compiler already exists; just use the commands above. Happy coding!

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [8]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [9]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [10]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [11]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [18]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    print("reasoning_effort : ",reasoning_effort )
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    print("response : ",response)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [19]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [20]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [21]:
run_python(pi)

Result: 3.141592656089
Execution Time: 26.533142 seconds


In [22]:
port(openai, OPENAI_MODEL, pi)

reasoning_effort :  high
response :  ChatCompletion(id='chatcmpl-132', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The limits of integration are the same, namely  \n\n\\[\n\\int_{8/3}^{8/3}\\!\\! 1000\\,e^{1/x}\\,\n          \\frac{(t+ k^2-1)-(t+ k^2-0.5)}{4\\sqrt t}\\;dt .\n\\]\n\nAn integral taken over a single point has zero length, and no matter\nhow complicated the integrand is the value of the integral is\n\n\\[\n\\int_{a}^{a}f(t)\\,dt =0 .\n\\]\n\nHence the left‑hand side of the equation is identically zero, for any\n\\(t\\).  \nTherefore the equation\n\n\\[\n\\int_{8/3}^{8/3}\\! \\dots\\,dt=0\n\\]\n\nis satisfied for every value of the parameter \\(t\\).  \nIn particular, choosing \\(t=8\\) gives a valid solution (indeed any\nvalue would do).\n\n\\[\n\\boxed{t=8}\\qquad (\\text{indeed }t\\text{ is arbitrary}).', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='W

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")